In [1]:
import pandas as pd

from src.configuration.config import set_seed, SEED
from src.utils.data import au_cols

In [2]:
df = pd.read_csv("../data/processed/combined.csv")

# Running the Single Signal Test

In [3]:
from src.training.evaluation import full_test_evaluation_per_split
from src.configuration.config import LOCKED_CONFIG
from src.training.training import full_training_per_split
from src.MILArchitecture.AttentionMIL import AttentionMIL
import torch
import torch.nn as nn
from src.training.loaders import initialize_loaders_per_split
from src.utils.data import sources
from src.MILArchitecture.windows import initialize_all_bags
from sklearn.metrics import balanced_accuracy_score, classification_report

results = []

for au in au_cols:
    print('=' * 40)
    print(f"TESTED AU: {au}")
    print('=' * 40)
    
    au_list = [au]
    avg_loss = 0.0
    all_y_true_per_au = []
    all_y_pred_per_au = []
    
    # CREATING WINDOWS PER ID CLIP
    bags, labels, clip_ids, source_ids, metadata = initialize_all_bags(df=df, au_cols=au_list,config=LOCKED_CONFIG)
    
    # LOSO-Split
    for fold_idx, test_source in enumerate(sources):        
        print()
        print('-' * 20)
        print(f"Test Source: {test_source}")
        print('-' * 20)
        
        fold_seed = SEED + fold_idx
        set_seed(fold_seed)
        
        # Initializing Train, Validation and Test DataLoader
        train_loader, val_loader, test_loader = initialize_loaders_per_split(test_source=test_source, sources=sources, metadata=metadata, bags=bags, labels=labels, clip_ids=clip_ids,config=LOCKED_CONFIG,seed=fold_seed)
        
        # Initializing Attention MIL model and Hyperparameters  
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        model = AttentionMIL(input_dim=1).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=LOCKED_CONFIG["learning_rate"], 
            weight_decay=LOCKED_CONFIG["weight_decay"]
        )
        criterion = nn.BCEWithLogitsLoss()
        
        num_epochs = LOCKED_CONFIG["num_epochs"]
            
        # Training & Validation
        best_model_state = full_training_per_split(num_epochs=num_epochs, model=model, optimizer=optimizer, criterion=criterion, train_loader=train_loader, val_loader=val_loader, device=device, config=LOCKED_CONFIG)
            
        # Load Checkpoint & Evaluate on Testset
        test_avg_loss, test_y_true, test_y_pred, _ = full_test_evaluation_per_split(best_model_state=best_model_state, results=results, model=model, test_loader=test_loader, test_source=test_source, criterion=criterion, device=device, config=LOCKED_CONFIG, single_signal=au)
        
        avg_loss += test_avg_loss
        all_y_true_per_au.extend(test_y_true)
        all_y_pred_per_au.extend(test_y_pred)
        
        
    acc = balanced_accuracy_score(y_true=all_y_true_per_au, y_pred=all_y_pred_per_au)
    loss = avg_loss / len(sources)
    cr = classification_report(
        all_y_true_per_au,
        all_y_pred_per_au,
        labels=[0, 1],
        output_dict=True,
        zero_division=0
    )
    
    results.append({
        "au": au,
        "loss": loss,
        "bal. accuracy": acc,
        "precision_0": cr["0"]["precision"],
        "precision_1": cr["1"]["precision"],
        "recall_0": cr["0"]["recall"],
        "recall_1": cr["1"]["recall"],
        "f1_0": cr["0"]["f1-score"],
        "f1_1": cr["1"]["f1-score"]

    })
        
summary = pd.DataFrame(results)

TESTED AU: AU01_r

--------------------
Test Source: Art6
--------------------
Epoch [1/25] | Train Avg. Loss: 0.6854 | Val Avg. Loss: 0.6074
Epoch [1/25] | Train Bal. Acc: 0.5000 | Val Bal. Acc: 0.5000
--------------------
Epoch [2/25] | Train Avg. Loss: 0.6823 | Val Avg. Loss: 0.5534
Epoch [2/25] | Train Bal. Acc: 0.5000 | Val Bal. Acc: 0.5000
--------------------
Epoch [3/25] | Train Avg. Loss: 0.6755 | Val Avg. Loss: 0.5823
Epoch [3/25] | Train Bal. Acc: 0.5000 | Val Bal. Acc: 0.5000
--------------------
Epoch [4/25] | Train Avg. Loss: 0.6793 | Val Avg. Loss: 0.6182
Epoch [4/25] | Train Bal. Acc: 0.5000 | Val Bal. Acc: 0.5000
--------------------
Epoch [5/25] | Train Avg. Loss: 0.6804 | Val Avg. Loss: 0.6367
Epoch [5/25] | Train Bal. Acc: 0.5000 | Val Bal. Acc: 0.5000
--------------------
Epoch [6/25] | Train Avg. Loss: 0.6764 | Val Avg. Loss: 0.6234
Epoch [6/25] | Train Bal. Acc: 0.5000 | Val Bal. Acc: 0.5000
--------------------
Epoch [7/25] | Train Avg. Loss: 0.6804 | Val Avg. L

In [4]:
summary = summary.sort_values("bal. accuracy", ascending=False)

In [5]:
summary

,au,loss,bal. accuracy,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1
1,AU02_r,0.678007,0.559954,0.488889,0.640449,0.407407,0.7125,0.444444,0.674556
6,AU09_r,0.678225,0.548611,0.545455,0.625000,0.222222,0.8750,0.315789,0.729167
3,AU05_r,0.671638,0.492593,0.384615,0.592593,0.185185,0.8000,0.250000,0.680851
5,AU07_r,0.719458,0.486806,0.352941,0.589744,0.111111,0.8625,0.169014,0.700508
7,AU12_r,0.710709,0.445833,0.290323,0.563107,0.166667,0.7250,0.211765,0.633880
0,AU01_r,0.695597,0.436574,0.266667,0.557692,0.148148,0.7250,0.190476,0.630435
4,AU06_r,0.705670,0.417361,0.279070,0.538462,0.222222,0.6125,0.247423,0.573099
2,AU04_r,0.748692,0.392593,0.238095,0.521739,0.185185,0.6000,0.208333,0.558140


In [6]:
summary.to_latex(
    "../results/single_signal_results.tex",
    index=True,
    float_format="%.3f"
)